# Output-free author training recipe
Derived from the corrected `train_yolo26s_object_detection_on_custom_dataset2.ipynb`. All outputs, execution counts, attachments and Colab metadata are removed. No prescription images or patient records are included.
Historical recipe only: dataset-version linkage to the released checkpoint is author-reported, not proven by this notebook. Use private, authorized dataset access. Training reruns are not claimed byte-identical. Pin Ultralytics 8.4.21 to match the logged training environment before running the historical cells.
See MODEL_CARD.md, AUTHOR_DISTRIBUTION_DECISION.md and ARTIFACT_MANIFEST.json for current artifact identity, licences and verification scope.


[![Roboflow Notebooks](https://media.roboflow.com/notebooks/template/bannertest2-2.png?ik-sdk-version=javascript-1.4.3&updatedAt=1672932710194)](https://github.com/roboflow/notebooks)

# Prescription Medicine Region Detection — YOLO26 Object Detection

---

[![GitHub](https://badges.aleen42.com/src/github.svg)](https://github.com/ultralytics/ultralytics)

This notebook fine-tunes **YOLO26** for **object detection on prescription images** to detect the following regions:

| Class | Description |
|---|---|
| `Date` | Date written on the prescription |
| `Frequency` | Dosage frequency (e.g. twice daily) |
| `Lines` | General text lines on the prescription |
| `Medication` | Prescribed medicine name |
| `Passport-as2v` | Passport / patient ID area |
| `Pilgrim` | Pilgrim-specific identifier area |
| `Quantity` | Quantity / amount prescribed |

The dataset is uploaded directly to the Colab runtime as a zip file — **no Roboflow API key required**.


## Setup

### Before you start

Let's make sure that we have access to GPU. We can use `nvidia-smi` command to do that. In case of any problems navigate to `Runtime` -> `Change runtime type` -> `Hardware accelerator`, set it to `GPU`, and then click `Save`.

In [ ]:
!nvidia-smi

**NOTE:** To make it easier for us to manage datasets, images and models we create a `HOME` constant.

In [ ]:
import os
HOME = os.getcwd()
print(HOME)

### Install dependencies required for YOLO26

In [ ]:
%pip install -q "ultralytics>=8.4.0" supervision

# prevent ultralytics from tracking your activity
!yolo settings sync=False
import ultralytics
ultralytics.checks()


## Fine-tune YOLO26 on Prescription Dataset


### Upload Dataset

Upload your YOLO dataset zip to the Colab runtime, then extract it.

> **How to prepare:** Zip the entire `Prescription.v1-version-01_test.yolo26` folder (the one containing `data.yaml`, `train/`, `valid/`, `test/`) and upload it below.
>
> Alternatively, if the dataset is stored in **Google Drive**, use the Drive mount option (Option B) instead.


In [ ]:
# ── Option A: Upload dataset zip directly to Colab runtime ──────────────────
import os
import zipfile
from google.colab import files

os.makedirs(f'{HOME}/datasets', exist_ok=True)

# print("⬆  Upload your dataset zip file (e.g. Prescription.v1-version-01_test.yolo26.zip):")
# uploaded = files.upload()

zip_filename = os.path.join(f'{HOME}/datasets', 'Prescription.v2-prescription-v2.yolo26.zip')
print(f"\n📦 Extracting '{zip_filename}' ...")

with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall(f'{HOME}/datasets')

print("✅ Extraction complete.")

# Auto-detect the extracted dataset folder (contains data.yaml)
DATASET_PATH = None
for root, dirs, filenames in os.walk(f'{HOME}/datasets'):
    if 'data.yaml' in filenames:
        DATASET_PATH = root
        break

if DATASET_PATH is None:
    raise FileNotFoundError("Could not find data.yaml inside the extracted archive. "
                            "Make sure the zip contains a data.yaml at its root or one level deep.")

print(f"\n📂 Dataset path: {DATASET_PATH}")
!cat {DATASET_PATH}/data.yaml


In [ ]:
# ── Option B: Load dataset from Google Drive (skip if you used Option A) ─────
# Uncomment and run this cell INSTEAD of Option A if your dataset lives on Drive.

# from google.colab import drive
# import shutil, os
#
# drive.mount('/content/drive')
#
# # Update this path to where the dataset folder sits in your Drive
# DRIVE_DATASET_PATH = '/content/drive/MyDrive/Prescription.v1-version-01_test.yolo26'
#
# DATASET_PATH = f'{HOME}/datasets/Prescription.v1-version-01_test.yolo26'
# shutil.copytree(DRIVE_DATASET_PATH, DATASET_PATH)
# print(f"Dataset copied to: {DATASET_PATH}")


## Training

Train YOLO26 on the prescription dataset. Adjust `epochs` and `imgsz` as needed.

- **Model:** `yolo26n.pt` (nano) — swap to `yolo26s.pt` / `yolo26m.pt` for more capacity.
- **Epochs:** 50 is a good starting point for this dataset size.
- **imgsz:** 640 works well; increase to 1280 if prescriptions contain fine text.


In [ ]:
%cd {HOME}

!yolo task=detect mode=train \
    model=yolo26s.pt \
    data={DATASET_PATH}/data.yaml \
    epochs=40 \
    imgsz=640 \
    plots=True \
    name=prescription_det


**NOTE:** Training results are saved to `{HOME}/runs/detect/prescription_det/`. Let's inspect them.


In [ ]:
!ls {HOME}/runs/detect/prescription_det/


In [ ]:
from IPython.display import Image as IPyImage

IPyImage(filename=f'{HOME}/runs/detect/prescription_det/confusion_matrix.png', width=600)


In [ ]:
from IPython.display import Image as IPyImage

IPyImage(filename=f'{HOME}/runs/detect/prescription_det/results.png', width=600)


In [ ]:
from IPython.display import Image as IPyImage

IPyImage(filename=f'{HOME}/runs/detect/prescription_det/val_batch0_pred.jpg', width=600)


## Validate fine-tuned model

In [ ]:
!yolo task=detect mode=val \
    model={HOME}/runs/detect/prescription_det/weights/best.pt \
    data={DATASET_PATH}/data.yaml


## Save / Export Trained Model

Download the best weights from the Colab runtime, or copy them to Google Drive.


In [ ]:
BEST_WEIGHTS = f'{HOME}/runs/detect/prescription_det/weights/best.pt'

# ── Option A: Download to local machine ─────────────────────────────────────
from google.colab import files
files.download(BEST_WEIGHTS)

# ── Option B: Copy to Google Drive (uncomment to use) ───────────────────────
from google.colab import drive
drive.mount('/content/drive')
import shutil
shutil.copy(BEST_WEIGHTS, '/content/drive/MyDrive/prescription_det_best.pt')
print("Saved to Google Drive.")


## Inference with Fine-tuned Model

Run the trained model on the **test set** to see how well it detects prescription regions.


### CLI — Predict on Test Images


In [ ]:
!yolo task=detect mode=predict \
    model={HOME}/runs/detect/prescription_det/weights/best.pt \
    source={DATASET_PATH}/test/images \
    save=True \
    verbose=False


### SDK — Visualise Predictions on Test Set


In [ ]:
from ultralytics import YOLO

model = YOLO(f'{HOME}/runs/detect/prescription_det/weights/best.pt')
print("Model loaded. Classes:", model.names)


In [ ]:
import supervision as sv

ds_test = sv.DetectionDataset.from_yolo(
    images_directory_path=f"{DATASET_PATH}/test/images",
    annotations_directory_path=f"{DATASET_PATH}/test/labels",
    data_yaml_path=f"{DATASET_PATH}/data.yaml",

)
print(f"Test set: {len(ds_test)} images — Classes: {ds_test.classes}")


In [ ]:
import supervision as sv
from PIL import Image

# 7 colours — one per prescription class:
# Date, Frequency, Lines, Medication, Passport-as2v, Pilgrim, Quantity
PALETTE = sv.ColorPalette.from_hex([
    "#FF6B6B",  # Date        — soft red
    "#FFD93D",  # Frequency   — amber
    "#6BCB77",  # Lines       — green
    "#4D96FF",  # Medication  — blue
    "#C77DFF",  # Passport    — purple
    "#FF9A3C",  # Pilgrim     — orange
    "#00C9A7",  # Quantity    — teal
])

CLASS_NAMES = ['Date', 'Frequency', 'Lines', 'Medication', 'Passport-as2v', 'Pilgrim', 'Quantity']


def annotate(image: Image.Image, detections: sv.Detections) -> Image.Image:
    text_scale = sv.calculate_optimal_text_scale(resolution_wh=image.size)

    box_annotator = sv.BoxAnnotator(color=PALETTE, thickness=2)
    label_annotator = sv.LabelAnnotator(
        color=PALETTE,
        text_color=sv.Color.WHITE,
        text_scale=text_scale,
        smart_position=True
    )

    labels = [CLASS_NAMES[int(cls)] for cls in detections.class_id] if detections.class_id is not None else []

    out = image.copy()
    out = box_annotator.annotate(out, detections)
    out = label_annotator.annotate(out, detections, labels=labels)
    out.thumbnail((1000, 1000))
    return out


In [ ]:
import random
import matplotlib.pyplot as plt

N = min(9, len(ds_test))   # show up to 9 test images
L = len(ds_test)

annotated_images = []

for i in random.sample(range(L), N):
    path, _, annotations = ds_test[i]
    image = Image.open(path).convert("RGB")
    result = model.predict(image, verbose=False)[0]
    detections = sv.Detections.from_ultralytics(result)
    annotated_image = annotate(image, detections)
    annotated_images.append(annotated_image)

cols = 3
rows = (N + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(14, 14 * rows // cols))

for ax, img in zip(axes.flat, annotated_images):
    ax.imshow(img)
    ax.axis("off")

# hide any unused subplots
for ax in list(axes.flat)[N:]:
    ax.set_visible(False)

plt.suptitle("Prescription Detection — Test Set Predictions", fontsize=14, y=1.01)
plt.subplots_adjust(wspace=0.02, hspace=0.02, left=0.01, right=0.99, top=0.99, bottom=0.01)
plt.show()


In [ ]:
# Show the first annotated prescription in detail
annotated_images[0]


<div align="center">
  <p>
    <strong>Medora — Prescription Region Detection</strong><br/>
    Trained with <a href="https://github.com/ultralytics/ultralytics">YOLO26</a> &amp;
    <a href="https://github.com/roboflow/supervision">Supervision</a>
  </p>
  <p>
    Dataset classes: <code>Date</code> · <code>Frequency</code> · <code>Lines</code> ·
    <code>Medication</code> · <code>Passport-as2v</code> · <code>Pilgrim</code> · <code>Quantity</code>
  </p>
</div>
